In [1]:
from google.colab import files
uploaded = files.upload()


Saving haunted_places.json to haunted_places.json


In [ ]:
import json
import pprint

# Open the JSON file
with open('haunted_places.json', 'r') as f:
    data = json.load(f)

# Check type to understand structure
print(type(data))

# Print first few rows depending on structure

# If it's a list (common case)
if isinstance(data, list):
    pprint.pprint(data[:5])

# If it's a dictionary with a key like "places" or similar
elif isinstance(data, dict):
    for key in data:
        print(f"Top-level key: {key} → type: {type(data[key])}")
    # Then for example, if it's under 'places':
    if 'places' in data:
        pprint.pprint(data['places'][:5])


<class 'list'>
[{'Alcohol deaths per capita': '5.6',
  'Annual deaths attributable to excessive alcohol use': '4548.0',
  'Apparition Type': 'Ghost',
  'Audio Evidence': 'True',
  'Daylight Duration': '11:49',
  'Daylight Duration 2': '11:44',
  'Event Type': 'Murder',
  'Ghost Category': 'Male',
  'Haunted Places Date': '3/12/1954 0:00',
  'Haunted Places Witness Count': '1.0',
  'Housing Units': '31.0',
  'Image/Video/Visual Evidence': 'True',
  'Median Age': '60.1',
  'NER_ENTITIES': "[('Ada witch -', 'ORG'), ('3-mile', 'QUANTITY'), ('the Ada "
                  "Cemetery', 'FAC'), ('Egypt Valley', 'ORG'), ('Honey Creek', "
                  "'GPE'), ('late one night', 'TIME'), ('late at night', "
                  "'TIME'), ('the night', 'TIME'), ('the turn of the 20th "
                  "century', 'DATE'), ('Honeycreek Road', 'FAC'), ('Seidman "
                  "Park', 'FAC'), ('Findlay Cemetery', 'PERSON'), ('Ghosts of "
                  "Grand Rapids', 'WORK_OF_ART')]",
  'N

In [3]:
import pandas as pd
import json
from datetime import datetime

# Assuming the data is in a JSON file
# Load your JSON data into a pandas DataFrame
with open('haunted_places.json') as f:
    haunted_places = json.load(f)

# Convert JSON to DataFrame for easier manipulation
df = pd.json_normalize(haunted_places)

# Print the column names to check for discrepancies
print("Columns in the DataFrame:", df.columns)

# Print the first few rows to inspect the data
print(df.head())

# Function to parse the date and extract the year
def parse_year(date_str):
    try:
        # Try parsing the date in different formats
        if isinstance(date_str, str):
            return datetime.strptime(date_str, "%m/%d/%Y %H:%M").year
        else:
            return None
    except:
        return None

# Apply the function to extract the year from the date
df['year'] = df['Haunted Places Date'].apply(parse_year)

# Check if 'Apparition Type' column exists
if 'Apparition Type' in df.columns:
    # Group by year and apparition type, then count occurrences
    apparition_counts = df.groupby(['year', 'Apparition Type']).size().unstack(fill_value=0)

    # Reset index to flatten the DataFrame
    apparition_counts = apparition_counts.reset_index()

    # Display the results
    print(apparition_counts)
else:
    print("Error: 'Apparition Type' column is missing.")


Columns in the DataFrame: Index(['city', 'country', 'description', 'location', 'state', 'state_abbrev',
       'longitude', 'latitude', 'city_longitude', 'city_latitude',
       'Audio Evidence', 'Image/Video/Visual Evidence', 'Haunted Places Date',
       'Haunted Places Witness Count', 'Time of Day', 'Apparition Type',
       'Ghost Category', 'Event Type',
       'Over 18 binge drink at least once per month',
       'Alcohol deaths per capita',
       'Annual deaths attributable to excessive alcohol use', 'Sunrise',
       'Sunset', 'Daylight Duration', 'Daylight Duration 2', 'crime_solved',
       'victim_sex', 'weapon', 'Median Age', 'Population', 'Housing Units',
       'moon_phase', 'moon_diameter', 'moon_distance', 'geotopic_name',
       'geotopic_longitude', 'geotopic_latitude', 'NER_ENTITIES',
       'NER_PERSONS', 'NER_ORGS', 'NER_LOCATIONS', 'image_path',
       'tika_caption'],
      dtype='object')
      city        country                                        descript

In [4]:
# Check for missing or NaN values in the 'year' or 'Apparition Type' columns
print(df[df['year'].isna()])  # Rows where 'year' is missing
print(df[df['Apparition Type'].isna()])  # Rows where 'Apparition Type' is missing

# Drop rows with NaN years or apparition types (if any)
df_cleaned = df.dropna(subset=['year', 'Apparition Type'])

# Re-run the grouping and counting after cleaning the data
apparition_counts_cleaned = df_cleaned.groupby(['year', 'Apparition Type']).size().unstack(fill_value=0).reset_index()

# Display the cleaned results
print(apparition_counts_cleaned)


                  city        country  \
3               Adrian  United States   
6      Algoma Township  United States   
12              Alpena  United States   
13           Ann Arbor  United States   
16             Atlanta  United States   
...                ...            ...   
10985         Thornton  United States   
10986       Walsenburg  United States   
10987      Westminster  United States   
10988      Westminster  United States   
10991    Woodland Park  United States   

                                             description  \
3      In the 1970's, one room, room 211, in the old ...   
6      On a winding dirt road next to the Rogue River...   
12     A few witnesses heard footsteps and saw heads ...   
13     Story goes some time in the late 70s a student...   
16     The old cabin located on Camp 8 road is haunte...   
...                                                  ...   
10985  Riverdale Road is a circuitous and meandering ...   
10986  half of the building

In [5]:
# Save the cleaned data as a JSON file
apparition_counts_cleaned.to_json('/content/apparition_counts_cleaned.json', orient='records')


In [6]:
from google.colab import files
files.download('/content/apparition_counts_cleaned.json')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>